# T5 Fine-tuning for Text Simplification

This notebook fine-tunes T5-small on the WikiAuto dataset for text simplification.

**Before running:** Go to `Runtime > Change runtime type > T4 GPU`

**Output:** A fine-tuned model saved to `t5-simplifier/` that you download and drop into the project.

## Step 1 - Install dependencies

In [ ]:
!pip install -q transformers datasets sentencepiece accelerate

## Step 2 - Check GPU

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

## Step 3 - Load dataset

We use WikiAuto - a large dataset of (complex Wikipedia sentence, simple Wikipedia sentence) pairs.
We only use 50,000 training pairs to keep training time under 2 hours on a T4.

In [ ]:
from datasets import load_dataset

dataset = load_dataset('wiki_auto', 'auto_acl')
print(dataset)
print('\nSample entry:')
print(dataset['train'][0])

In [ ]:
# Inspect column names - they vary slightly by dataset version
print('Columns:', dataset['train'].column_names)
print('Sample:', dataset['train'][0])

In [ ]:
# Limit to 50k train, 2k validation to keep training fast
MAX_TRAIN = 50000
MAX_VAL   = 2000

train_data = dataset['train'].shuffle(seed=42).select(range(min(MAX_TRAIN, len(dataset['train']))))
val_data   = dataset['validation'].shuffle(seed=42).select(range(min(MAX_VAL, len(dataset['validation']))))

print(f'Train samples: {len(train_data)}')
print(f'Val samples:   {len(val_data)}')

## Step 4 - Load tokenizer

In [ ]:
from transformers import T5Tokenizer

MODEL_NAME = 't5-small'
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)

PREFIX = 'simplify: '
MAX_INPUT_LEN  = 128
MAX_TARGET_LEN = 128

## Step 5 - Tokenize

**IMPORTANT:** Check the column names from Step 3 output and adjust `COMPLEX_COL` and `SIMPLE_COL` below if needed.

In [ ]:
# Adjust these if your Step 3 showed different column names
COMPLEX_COL = 'normal'   # the complex/original sentence
SIMPLE_COL  = 'simple'   # the simplified sentence

# If the above names are wrong, try: 'source', 'target' or 'complex', 'simple'

def tokenize(batch):
    inputs = [PREFIX + text for text in batch[COMPLEX_COL]]
    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding='max_length'
    )
    labels = tokenizer(
        batch[SIMPLE_COL],
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding='max_length'
    )
    # Replace padding token id with -100 so loss ignores padding
    label_ids = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels['input_ids']
    ]
    model_inputs['labels'] = label_ids
    return model_inputs

tokenized_train = train_data.map(tokenize, batched=True, remove_columns=train_data.column_names)
tokenized_val   = val_data.map(tokenize, batched=True, remove_columns=val_data.column_names)

print('Tokenization done.')
print('Train shape:', tokenized_train.shape)
print('Val shape:  ', tokenized_val.shape)

## Step 6 - Load model and set up training

In [ ]:
from transformers import T5ForConditionalGeneration, TrainingArguments, Trainer, DataCollatorForSeq2Seq

model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

training_args = TrainingArguments(
    output_dir='./t5-simplifier',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=200,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    predict_with_generate=True,
    fp16=True,                      # faster training on GPU
    report_to='none',               # disable wandb
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

print('Model and trainer ready.')

## Step 7 - Train

Expected time on T4 GPU: ~60-90 minutes for 3 epochs on 50k samples.

In [ ]:
trainer.train()

## Step 8 - Save the model

In [ ]:
model.save_pretrained('./t5-simplifier')
tokenizer.save_pretrained('./t5-simplifier')
print('Model saved to ./t5-simplifier')

## Step 9 - Quick test before downloading

In [ ]:
def simplify(text):
    inputs = tokenizer('simplify: ' + text, return_tensors='pt', max_length=128, truncation=True)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    outputs = model.generate(
        inputs['input_ids'],
        attention_mask=inputs['attention_mask'],
        max_length=128,
        num_beams=4,
        early_stopping=True
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

test_sentences = [
    "The accumulation of greenhouse gases in the atmosphere has led to unprecedented changes in global climate patterns.",
    "Photosynthesis is the process by which plants convert light energy into chemical energy stored in glucose.",
    "The judicial system operates on the principle that all individuals are presumed innocent until proven guilty."
]

for s in test_sentences:
    print('ORIGINAL: ', s)
    print('SIMPLIFIED:', simplify(s))
    print()

## Step 10 - Download the model

This zips the model folder and downloads it to your computer.
The zip will be around **250MB**.

In [ ]:
!zip -r t5-simplifier.zip t5-simplifier/

from google.colab import files
files.download('t5-simplifier.zip')
print('Download started.')

## After downloading

1. Unzip `t5-simplifier.zip` into your project folder
2. In `simplify.py`, change the default model path from `t5-small` to `./t5-simplifier`
3. Also change the prefix from `summarize:` to `simplify:` in the `simplify_with_t5()` function
4. Run normally - the local fine-tuned model will be used instead of downloading from HuggingFace

That's it. Your tool now uses a model actually trained for simplification.